<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/4_anonymizing_text_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Anonymizing Text · Solutions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a
short comment. Your numbers will differ a little from run to run, so read the comments as "what
to look for", not as exact results.

Interview transcripts, open survey answers, e-mails and case files are full of names, addresses,
phone numbers and details that point to a specific person. Before you share such data, or sometimes
before you may even analyze it, those details have to go. Doing that by hand for hundreds of
documents is slow and error-prone; doing it fully automatically is risky. This notebook shows how to
combine the two sensibly.

In this notebook you will:

- learn the difference between anonymization and pseudonymization, and between direct and indirect
  identifiers
- generate test transcripts in English and Dutch with *known* personal details, so we can measure
  how much each method catches
- find structured identifiers (e-mail, phone, postcode, IBAN) with regular expressions
- find names, places and organizations with spaCy's named entity recognition
- replace what you found: redaction, consistent pseudonyms, or realistic surrogates
- measure recall and precision, and see why recall matters most
- see what no tool catches, and how to organize a human check

**Contents**

1. What counts as identifying?
2. Test transcripts with known answers
3. Rules: regular expressions
4. Models: named entity recognition
5. Measuring what you caught
6. Replacing: redaction, pseudonyms, surrogates
7. What tools miss: indirect identifiers
8. Tools and good practice

No GPU needed.

## 0. Setup

We use spaCy (Bonus 10 goes deeper into it) with its small English and Dutch models.

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q
!python -m spacy download nl_core_news_sm -q

import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import spacy

nlp_en = spacy.load("en_core_web_sm")
nlp_nl = spacy.load("nl_core_news_sm")
pd.set_option("display.max_colwidth", 120)
print("spaCy", spacy.__version__)

## 1. What counts as identifying?

**Direct identifiers** point to one person on their own: a name, an e-mail address, a phone number,
a home address, a bank account, a citizen service number (BSN).

**Indirect identifiers** (or quasi-identifiers) don't, but *combinations* do: "a dairy farmer in
her fifties, the only woman on the village council, with a daughter studying in Utrecht" may describe
exactly one person in the Netherlands. In text, these are the hard part.

Two terms that the GDPR treats very differently:

| | What it means | Still personal data? |
|---|---|---|
| **Pseudonymization** | identifiers replaced by codes (`PERSON_1`); a key linking codes to people exists somewhere | **yes**: the GDPR still applies |
| **Anonymization** | no reasonable way left to identify anyone, by you or anyone else | no |

Most "anonymized" research data is, strictly speaking, pseudonymized. That's fine, as long as you
handle it accordingly. Real anonymization of rich text is hard, and is never achieved by software
alone.

> Your data management plan and informed consent forms set the rules for your project. Your faculty's
> data steward can help you decide what level is needed.

## 2. Test transcripts with known answers

To *measure* how well a method works, we need texts where we know exactly where every personal detail
is. So we generate interview transcripts from templates and record the position and type of every
detail we put in. All names, addresses and numbers are made up.

The generator is long but simple; you don't need to read it in detail. What matters is that it
produces a `text` and a list of `gold` spans: `(start, end, type)` for every identifier.

In [ ]:
rng = random.Random(42)

FIRST = ["Jan", "Pieter", "Anouk", "Femke", "Sanne", "Willem", "Hendrik", "Marieke", "Joost", "Gerrit",
         "Ingrid", "Bram", "Lotte", "Kees", "Ria", "Henk", "Wilma", "Geert"]
LAST = ["de Vries", "Jansen", "Bakker", "Visser", "Smit", "Meijer", "de Boer", "Mulder", "de Groot",
        "Bos", "Vos", "Peters", "Hendriks", "van Dijk", "Dekker", "van den Berg"]
VILLAGES = ["Lunteren", "Barneveld", "Scherpenzeel", "Renswoude", "Otterlo", "Bennekom",
            "Kootwijkerbroek", "Wekerom", "Harskamp", "Garderen", "Stroe", "Voorthuizen"]
STREETS = ["Dorpsstraat", "Kerkweg", "Molenweg", "Beekstraat", "Schoolstraat", "Veenweg"]
ORGS = ["Rabobank", "LTO Nederland", "FrieslandCampina", "Provincie Gelderland", "ForFarmers"]
CITIES = {"en": ["The Hague", "Amsterdam", "Utrecht", "Brussels"], "nl": ["Den Haag", "Amsterdam", "Utrecht", "Brussel"]}
MONTHS = {"en": ["January", "February", "March", "April", "May", "June", "July", "August", "September",
                 "October", "November", "December"],
          "nl": ["januari", "februari", "maart", "april", "mei", "juni", "juli", "augustus", "september",
                 "oktober", "november", "december"]}
LETTERS = "ABCDEFGHJKLMNPRTVWXZ"

# Which slots are identifiers, and of which type. CITY and YEARS are deliberately NOT identifiers:
# a big city or a number of years doesn't point to anyone.
SLOT_TYPE = {"NAME": "PERSON", "FIRST": "PERSON", "OTHER": "PERSON", "OTHER_FIRST": "PERSON",
             "CHILD": "PERSON", "child": "PERSON", "OFFICIAL": "PERSON", "VILLAGE": "LOCATION",
             "STREET": "ADDRESS", "POSTCODE": "POSTCODE", "EMAIL": "EMAIL", "PHONE": "PHONE",
             "IBAN": "IBAN", "DOB": "DATE", "ORG": "ORG"}

TEMPLATES = {
    "en": ["My name is {NAME}. I have been farming near {VILLAGE} for {YEARS} years.",
           "We live at {STREET}, {POSTCODE} {VILLAGE}, right next to the nature reserve.",
           "If you have more questions, you can reach me at {EMAIL} or call {PHONE}.",
           "I was born on {DOB}, so I remember how things were before all these rules.",
           "My neighbor {OTHER} already sold his cows. {OTHER_FIRST} says he doesn't regret it.",
           "The compensation from {ORG} was paid into {IBAN}, but it was not nearly enough.",
           "My daughter {CHILD} studies in {CITY} and wants nothing to do with farming.",
           "honestly i talked to {child} about it last week and she said we should wait.",
           "People in {CITY} don't understand what a farm is.",
           "{OTHER_FIRST} and I went to the protest together, with half of {VILLAGE}.",
           "Mrs. {OFFICIAL} from the municipality told us to wait for the new plan.",
           "My wife is the only veterinarian in {VILLAGE} who still visits farms at night."],
    "nl": ["Mijn naam is {NAME}. Ik boer al {YEARS} jaar in de buurt van {VILLAGE}.",
           "We wonen aan de {STREET}, {POSTCODE} {VILLAGE}, vlak naast het natuurgebied.",
           "Als u nog vragen heeft, kunt u mailen naar {EMAIL} of bellen met {PHONE}.",
           "Ik ben geboren op {DOB}, dus ik weet nog hoe het was voor al die regels.",
           "Mijn buurman {OTHER} heeft zijn koeien al verkocht. {OTHER_FIRST} zegt dat hij geen spijt heeft.",
           "De vergoeding van {ORG} is gestort op {IBAN}, maar het was lang niet genoeg.",
           "Mijn dochter {CHILD} studeert in {CITY} en wil niets met de boerderij te maken hebben.",
           "eerlijk gezegd heb ik er vorige week met {child} over gepraat en zij zei dat we moesten wachten.",
           "Mensen in {CITY} begrijpen niet wat een boerderij is.",
           "{OTHER_FIRST} en ik zijn samen naar het protest gegaan, met half {VILLAGE}.",
           "Mevrouw {OFFICIAL} van de gemeente zei dat we op het nieuwe plan moesten wachten.",
           "Mijn vrouw is de enige dierenarts in {VILLAGE} die 's nachts nog bij boerderijen komt."],
}
QUESTIONS = {"en": ["Could you tell me more about that?", "How did that feel?", "And what happened then?",
                    "What do you think about the new rules?"],
             "nl": ["Kunt u daar meer over vertellen?", "Hoe voelde dat?", "En wat gebeurde er toen?",
                    "Wat vindt u van de nieuwe regels?"]}


def make_identity(lang):
    name, other = f"{rng.choice(FIRST)} {rng.choice(LAST)}", f"{rng.choice(FIRST)} {rng.choice(LAST)}"
    first, last = name.split(" ", 1)
    child = rng.choice(FIRST)
    domain = rng.choice(["gmail.com", "hotmail.com", "ziggo.nl", f"boerderij{last.split()[-1].lower()}.nl"])
    phone = rng.choice([f"06-{rng.randint(10**7, 10**8 - 1)}", f"06 {rng.randint(1000, 9999)} {rng.randint(1000, 9999)}",
                        f"+31 6 {rng.randint(10**7, 10**8 - 1)}", f"0318-{rng.randint(10**5, 10**6 - 1)}"])
    day, month, year = rng.randint(1, 28), rng.randint(1, 12), rng.randint(1950, 1985)
    dob = rng.choice([f"{day} {MONTHS[lang][month - 1]} {year}", f"{day:02d}-{month:02d}-{year}"])
    return {"NAME": name, "FIRST": first, "OTHER": other, "OTHER_FIRST": other.split()[0],
            "CHILD": child, "child": child.lower(), "OFFICIAL": (lambda l: l[0].upper() + l[1:])(rng.choice(LAST)),
            "VILLAGE": rng.choice(VILLAGES), "STREET": f"{rng.choice(STREETS)} {rng.randint(1, 120)}",
            "POSTCODE": f"{rng.randint(1000, 9999)}{rng.choice([' ', ''])}{rng.choice(LETTERS)}{rng.choice(LETTERS)}",
            "EMAIL": f"{first.lower()}.{last.replace(' ', '').lower()}@{domain}", "PHONE": phone,
            "IBAN": f"NL{rng.randint(10, 99)} {rng.choice(['RABO', 'INGB', 'ABNA', 'TRIO'])} "
                    f"{rng.randint(0, 9999):04d} {rng.randint(0, 9999):04d} {rng.randint(0, 99):02d}",
            "DOB": dob, "ORG": rng.choice(ORGS), "CITY": rng.choice(CITIES[lang]),
            "YEARS": str(rng.choice([12, 20, 25, 30, 35, 40]))}


def render(template, identity, offset):
    # Fill in a template, returning the text and the gold spans (shifted by offset)
    out, spans, pos = "", [], 0
    for m in re.finditer(r"\{(\w+)\}", template):
        out += template[pos:m.start()]
        value, slot = identity[m.group(1)], m.group(1)
        if slot in SLOT_TYPE:
            spans.append((offset + len(out), offset + len(out) + len(value), SLOT_TYPE[slot]))
        out += value
        pos = m.end()
    return out + template[pos:], spans


def make_transcript(lang):
    identity = make_identity(lang)
    chosen = [0] + sorted(rng.sample(range(1, len(TEMPLATES[lang])), 7))   # always the introduction
    text, gold = "", []
    for k in chosen:
        if text:
            text += f"I: {rng.choice(QUESTIONS[lang])}\n"
        text += "R: "
        sentence, spans = render(TEMPLATES[lang][k], identity, offset=len(text))
        text += sentence + "\n"
        gold += spans
    return text, gold


rows = []
for lang in ["en", "nl"]:
    for i in range(30):
        text, gold = make_transcript(lang)
        rows.append({"id": f"{lang}_{i:02d}", "lang": lang, "text": text, "gold": gold})
corpus = pd.DataFrame(rows)

print(corpus.loc[0, "text"])
print("Gold spans:", [(corpus.loc[0, "text"][s:e], t) for s, e, t in corpus.loc[0, "gold"]][:8])

In [ ]:
gold_counts = Counter(t for spans in corpus["gold"] for _, _, t in spans)
print(f"{len(corpus)} transcripts, {sum(gold_counts.values())} identifiers:")
print(pd.Series(gold_counts).sort_values(ascending=False).to_string())

## 3. Rules: regular expressions

Identifiers with a fixed *format* are best found with **regular expressions** (Day 1): e-mail
addresses, phone numbers, Dutch postcodes (`6708 PB`), IBANs (`NL91 ABNA 0417 1643 00`). Rules are
transparent, fast, and you know exactly what they do. The patterns below are deliberately readable
rather than complete.

In [ ]:
PATTERNS = {
    # name.part@domain.tld
    "EMAIL": r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b",
    # Dutch mobile (06-12345678, 06 1234 5678, +31 6 12345678) or landline (0318-123456)
    "PHONE": r"(?:\+31[\s-]?|\b0)(?:6[\s-]?\d{2}(?:[\s-]?\d{2}){3}|[1-9]\d{1,2}[\s-]?\d{6,7})(?!\d)",
    # four digits (not starting with 0), optional space, two capital letters
    "POSTCODE": r"\b[1-9]\d{3}\s?[A-Z]{2}\b",
    # NL + 2 check digits + 4-letter bank code + 10 digits, with or without spaces
    "IBAN": r"\bNL\d{2}\s?[A-Z]{4}(?:\s?\d{2,4}){3}\b",
}


def detect_regex(text, patterns=PATTERNS):
    return [(m.start(), m.end(), label) for label, p in patterns.items() for m in re.finditer(p, text)]


example = corpus.loc[0, "text"]
for s, e, label in detect_regex(example):
    print(f"{label:<9} {example[s:e]}")

Rules also make mistakes. A postcode pattern happily matches other things that look like a postcode:

In [ ]:
for s, e, label in detect_regex("In 2019 EU farmers protested. Our herd grew from 1200 to 1450 HF cows."):
    print(label, "->", repr("In 2019 EU farmers protested. Our herd grew from 1200 to 1450 HF cows."[s:e]))

"2019 EU" and "1450 HF" (Holstein-Friesian) aren't postcodes. Masking them costs a little
information; it doesn't leak anything. That asymmetry comes back in Section 5.

## 4. Models: named entity recognition

Names, places and organizations have no fixed format, so we need a model. spaCy's **named entity
recognizer** (NER) tags spans as `PERSON`, `GPE` (countries, cities, villages), `LOC` (other
locations), `FAC` (buildings, streets), `ORG`, and more. We keep the types that can identify someone.

In [ ]:
NER_TYPES = {"PERSON", "GPE", "LOC", "FAC", "ORG"}


def detect_ner(text, nlp):
    return [(e.start_char, e.end_char, e.label_) for e in nlp(text).ents if e.label_ in NER_TYPES]


doc = nlp_en(example)
for ent in doc.ents:
    print(f"{ent.label_:<9} {ent.text!r:<40} {'(kept)' if ent.label_ in NER_TYPES else '(ignored)'}")

Look closely at the labels: the small model sometimes gets the *type* wrong (a Dutch name tagged as
an organization), misses names, or tags things that aren't identifying at all (a big city). For masking, a
wrong type is a small problem (the name is still hidden); a miss is a big one.

`displacy` shows the entities in the text itself:

In [ ]:
from spacy import displacy
displacy.render(doc, style="ent", jupyter=True)

Rules and models complement each other, so we combine them. When two detected spans overlap, we merge
them into one.

In [ ]:
def merge_spans(spans):
    # Merge overlapping spans; the merged span keeps the type of its longest part
    merged = []
    for s, e, label in sorted(spans):
        if merged and s < merged[-1][1]:
            ps, pe, plabel = merged[-1]
            longest = plabel if (pe - ps) >= (e - s) else label
            merged[-1] = (ps, max(pe, e), longest)
        else:
            merged.append((s, e, label))
    return merged


def detect_pii(text, nlp):
    return merge_spans(detect_regex(text) + detect_ner(text, nlp))


for s, e, label in detect_pii(example, nlp_en):
    print(f"{label:<9} {example[s:e]!r}")

## 5. Measuring what you caught

Two measures, from Day 2:

- **Recall**: of all real identifiers, how many did we mask? A miss is a **leak**.
- **Precision**: of everything we masked, how much was really an identifier? A false alarm costs
  **information** (the analysis loses a word), but leaks nothing.

For anonymization, recall matters most. We count an identifier as caught only if *all* its characters
are masked: hiding "Pieter" but not "de Vries" still leaks. We also report partial catches.

In [ ]:
def evaluate(df, detector):
    caught_rows, n_pred, n_pred_ok, false_alarms = [], 0, 0, []
    for text, gold in zip(df["text"], df["gold"]):
        pred = merge_spans(detector(text))
        masked = np.zeros(len(text), dtype=bool)
        for s, e, _ in pred:
            masked[s:e] = True
        is_gold = np.zeros(len(text), dtype=bool)
        for s, e, label in gold:
            is_gold[s:e] = True
            chars = [i for i in range(s, e) if not text[i].isspace()]     # spaces don't leak anything
            caught_rows.append({"type": label, "text": text[s:e],
                                "caught": masked[chars].all(), "partly": masked[chars].any()})
        for s, e, label in pred:
            n_pred += 1
            if is_gold[s:e].any():
                n_pred_ok += 1
            else:
                false_alarms.append((text[s:e], label))
    caught = pd.DataFrame(caught_rows)
    by_type = caught.groupby("type")[["caught", "partly"]].mean().round(2)
    by_type["n"] = caught["type"].value_counts()
    summary = {"recall (strict)": float(caught["caught"].mean()), "recall (partial)": float(caught["partly"].mean()),
               "precision": n_pred_ok / max(n_pred, 1)}
    return by_type, summary, caught, false_alarms


english = corpus[corpus["lang"] == "en"]
for name, detector in [("regex only", detect_regex),
                       ("NER only", lambda t: detect_ner(t, nlp_en)),
                       ("regex + NER", lambda t: detect_pii(t, nlp_en))]:
    _, summary, _, _ = evaluate(english, detector)
    print(f"{name:<12}", "  ".join(f"{k}: {v:.2f}" for k, v in summary.items()))

In [ ]:
by_type, summary, caught, false_alarms = evaluate(english, lambda t: detect_pii(t, nlp_en))
by_type

In [ ]:
print("Missed (or only partly masked):")
display(caught[~caught["caught"]].drop_duplicates("text").head(12))
print("False alarms:", Counter(false_alarms).most_common(8))

Typical results with the small English model:

- the **regex types** are caught almost perfectly, because the generator uses the formats the patterns
  were written for (real data is messier);
- **names** are caught less reliably, with lowercase names (as speech recognition sometimes writes
  them) a classic miss;
- **addresses** are often only partly masked (the street but not the number, or the reverse);
- **dates of birth** aren't caught at all yet: that's Exercise 1;
- false alarms are mostly big cities and other harmless entities.

Your exact numbers depend on the spaCy version. Larger models (`en_core_web_lg`, or the transformer
model `en_core_web_trf`) catch more names, at the cost of speed.

> **✏️ Exercise 1**
>
> Dates of birth are identifiers too. Write a regular expression `DATE` that matches dates such as
> `12 March 1965`, `3 maart 1971` and `05-11-1958`, add it to a copy of `PATTERNS`, and evaluate the
> combined detector again on the English transcripts. What happens to date recall, and to precision?

In [ ]:
# ✅ Solution
MONTH_NAMES = "|".join(MONTHS["en"] + MONTHS["nl"])
PATTERNS_DATE = dict(PATTERNS, DATE=rf"\b\d{{1,2}}\s(?:{MONTH_NAMES})\s(?:19|20)\d{{2}}\b|\b\d{{1,2}}-\d{{1,2}}-(?:19|20)\d{{2}}\b")

def detect_pii_dates(text, nlp):
    return merge_spans(detect_regex(text, PATTERNS_DATE) + detect_ner(text, nlp))

by_type, summary, _, _ = evaluate(english, lambda t: detect_pii_dates(t, nlp_en))
display(by_type)
print({k: round(v, 2) for k, v in summary.items()})
# Date recall goes to (nearly) 1 and precision barely moves, because the pattern is specific: it
# needs a day, a month and a four-digit year. Note the double braces {{1,2}} inside an f-string.

> **✏️ Exercise 2**
>
> Evaluate `detect_pii` with the **Dutch** model (`nlp_nl`) on the Dutch transcripts. Compare recall
> per type, and precision, with the English results. Then look at the misses *and* the false alarms.
> Can you fix the most common false alarm?

In [ ]:
# ✅ Solution
dutch = corpus[corpus["lang"] == "nl"]
by_type_nl, summary_nl, caught_nl, false_nl = evaluate(dutch, lambda t: detect_pii(t, nlp_nl))
by_type_en, summary_en, _, _ = evaluate(english, lambda t: detect_pii(t, nlp_en))

display(pd.concat({"English": by_type_en["caught"], "Dutch": by_type_nl["caught"]}, axis=1))
print("Dutch:", {k: round(v, 2) for k, v in summary_nl.items()})
print("Most common false alarms:", Counter(false_nl).most_common(5))
display(caught_nl[~caught_nl["caught"]].drop_duplicates("text").head(10))

# Fix: ignore detections that are just a speaker label ("I", "R") at the start of a line
def detect_pii_transcript(text, nlp):
    return [(s, e, label) for s, e, label in detect_pii(text, nlp)
            if not (text[s:e] in {"I", "R"} and (s == 0 or text[s - 1] == "\n"))]

_, summary_fixed, _, _ = evaluate(dutch, lambda t: detect_pii_transcript(t, nlp_nl))
print("Dutch, speaker labels ignored:", {k: round(v, 2) for k, v in summary_fixed.items()})
# Look at your own numbers, but in our run the small Dutch model did WORSE than the English one:
# it missed more names (first names and surnames on their own especially), only ever masked the
# street of an address and not the house number, and tagged the interviewer label "I" as a person
# about a hundred times. Speaker labels are in almost every transcript, so handle them explicitly.
# The lesson: never trust a tool's reputation (or one overall number) for your language and text
# type. Measure it on your own material and read the misses.

### Finding a name once, masking it everywhere

A useful trick: once the detector has found a name *somewhere* in a document, search for every other
occurrence of it in that document, ignoring case. The model may miss "Joost says..." or a lowercase
"marieke" on their own, but it doesn't need to find them if it found "Joost Mulder" or "Marieke"
elsewhere in the text.

> **✏️ Exercise 3**
>
> Write `propagate_names(text, spans)`: for every detected `PERSON` span (and, since the small models
> often call Dutch names organizations, every `ORG` span of two or three words), find all other
> occurrences in the text of the full name *and* of its first word, ignoring case, and add them as
> extra spans. Skip words shorter than three letters. How much does person recall improve on the English
> transcripts? Does precision suffer?

In [ ]:
# ✅ Solution
def propagate_names(text, spans):
    extra = []
    for s, e, label in spans:
        name = text[s:e]
        words = name.split()
        if label == "PERSON" or (label == "ORG" and 2 <= len(words) <= 3):
            for variant in {name, words[0]}:
                if len(variant) < 3:
                    continue
                for m in re.finditer(rf"\b{re.escape(variant)}\b", text, flags=re.IGNORECASE):
                    extra.append((m.start(), m.end(), label))
    return merge_spans(spans + extra)

for name, detector in [("regex + NER", lambda t: detect_pii(t, nlp_en)),
                       ("+ propagation", lambda t: propagate_names(t, detect_pii(t, nlp_en)))]:
    by_type, summary, _, _ = evaluate(english, detector)
    print(f"{name:<14} person recall {by_type.loc['PERSON', 'caught']:.2f}  ",
          "  ".join(f"{k}: {v:.2f}" for k, v in summary.items()))
# Propagation catches the repeated first names and the lowercase versions, as long as the name was
# found once. It can over-mask: an organization's first word ("Provincie") gets masked everywhere.
# For pseudonyms you'd also want "Joost" to get the same code as "Joost Mulder"; that needs a small
# lookup from first words to full names, built the same way.

## 6. Replacing: redaction, pseudonyms, surrogates

Once you know *where* the identifiers are, there are three common ways to replace them:

| Strategy | Example | Pros | Cons |
|---|---|---|---|
| **Redaction** | `[PERSON]` | simple, clearly marked | loses who-is-who: are two `[PERSON]`s the same? |
| **Consistent pseudonyms** | `[PERSON_1]`, `[PERSON_2]` | keeps who-is-who | needs a key (store it safely, or destroy it) |
| **Surrogates** | `Jan Jansen` → `Kees Mulder` | text reads naturally | a missed real name hides among fake ones |

First a general function that replaces spans, then the first two strategies:

In [ ]:
COARSE = {"PERSON": "PERSON", "GPE": "LOCATION", "LOC": "LOCATION", "FAC": "LOCATION", "ORG": "ORG"}


def replace_spans(text, spans, replacement):
    # Replace each span with replacement(original_text, type), working left to right
    out, last = [], 0
    for s, e, label in merge_spans(spans):
        out.append(text[last:s])
        out.append(replacement(text[s:e], COARSE.get(label, label)))
        last = e
    out.append(text[last:])
    return "".join(out)


def redact(value, label):
    return f"[{label}]"


class Pseudonymizer:
    # Gives every distinct value a stable code, per type: [PERSON_1], [PERSON_2], ...
    def __init__(self):
        self.codes, self.counts = {}, Counter()

    def __call__(self, value, label):
        key = (label, value.lower())
        if key not in self.codes:
            self.counts[label] += 1
            self.codes[key] = f"[{label}_{self.counts[label]}]"
        return self.codes[key]


spans = detect_pii(example, nlp_en)
print(replace_spans(example, spans, redact))
pseudo = Pseudonymizer()
print(replace_spans(example, spans, pseudo))

One `Pseudonymizer` used for the whole corpus gives the same person the same code in every
transcript, which keeps networks and repeated mentions analyzable. Its `codes` dictionary **is the
key**: anyone who has it can undo the pseudonymization. Store it apart from the data (encrypted, with
restricted access), or delete it if you'll never need to go back.

In [ ]:
pseudo = Pseudonymizer()
corpus["pseudonymized"] = [replace_spans(t, detect_pii(t, nlp_en if lang == "en" else nlp_nl), pseudo)
                           for t, lang in zip(corpus["text"], corpus["lang"])]
key = pd.DataFrame([(label, value, code) for (label, value), code in pseudo.codes.items()],
                   columns=["type", "original", "code"])
print(corpus.loc[1, "pseudonymized"])
print(f"The key has {len(key)} entries, e.g.:")
key.head()

> **✏️ Exercise 4**
>
> Write a **surrogate** replacement: persons get a made-up name, locations a made-up village, and
> every other type a redaction tag. Make it consistent (the same original always gets the same
> surrogate). Apply it to a transcript and read the result. Why might surrogates make it *harder*
> to spot a name the detector missed?

In [ ]:
# ✅ Solution
FAKE_NAMES = ["Tom Hiddema", "Lisa Overbeek", "Ruud Klaassen", "Eline Stam", "Bart Zijlstra",
              "Noor Wiersma", "Sven Brink", "Iris Kok"]
FAKE_PLACES = ["Westerbroek", "Oosterveld", "Nieuwland", "Hoogdorp", "Laagwater"]

class Surrogates:
    def __init__(self, seed=0):
        self.map, self.rng = {}, random.Random(seed)
    def __call__(self, value, label):
        key = (label, value.lower())
        if key not in self.map:
            pool = {"PERSON": FAKE_NAMES, "LOCATION": FAKE_PLACES}.get(label)
            self.map[key] = self.rng.choice(pool) if pool else f"[{label}]"
        return self.map[key]

text = english["text"].iloc[3]
print(replace_spans(text, detect_pii(text, nlp_en), Surrogates()))
# The text reads naturally, which helps readers and some analyses. But look closely: a name the
# detector missed now sits among the fake ones, and nobody can tell which names are real ("hiding in
# plain sight", Carrell et al. 2013). That cuts both ways: it also makes a leak harder for an
# outsider to find. And when the NER got the TYPE wrong, the surrogate is absurd (a village replaced
# by a person's name). With redaction tags, every unmasked name stands out during a check.

## 7. What tools miss: indirect identifiers

Look at this sentence from one of the pseudonymized transcripts:

In [ ]:
for t in corpus["pseudonymized"]:
    hits = [line for line in t.splitlines() if "only veterinarian" in line or "enige dierenarts" in line]
    if hits:
        print(hits[0]); break

The village is masked, but "my wife is the only veterinarian who still visits farms at night"
remains, and together with other details in the same interview (how long they've farmed, a daughter
studying in a city, a protest) it may well point to one family. No pattern or entity recognizer will
flag this, because nothing in it is a name.

What helps:

- **Generalize** instead of masking: "a village in Gelderland", "in her fifties", "a large dairy farm".
- **Read the transcripts** with the question "could someone who knows this area recognize the
  person?". This is the human part that can't be automated away.
- Consider *who* could re-identify: the respondent's neighbors usually know far more than an outside
  attacker.
- Share less: aggregated or coded data instead of full transcripts, or access on request.

Software helps with the human check too: list what *could* still be a leak, and let a person decide.

> **✏️ Exercise 5**
>
> Write a function `leak_candidates(text)` that lists capitalized words in a pseudonymized text that are
> **not** at the start of a sentence, not inside a code like `[PERSON_1]`, and not in a small allow
> list (`I`, `R`, `Mr.`, `Mrs.`, ...). Run it over all pseudonymized English transcripts and count
> what it finds. Which are real leaks, and which are false alarms?

In [ ]:
# ✅ Solution
ALLOW = {"I", "R", "Mr", "Mrs", "Mevrouw", "Meneer", "U"}

def leak_candidates(text):
    candidates = []
    for line in text.splitlines():
        line = re.sub(r"\[[A-Z]+_\d+\]", "", line)              # drop the codes
        for sentence in re.split(r"(?<=[.!?:])\s+", line):
            words = re.findall(r"\b[A-Z][\w'-]*", sentence)
            first = re.match(r"\s*([A-Z][\w'-]*)", sentence)
            for w in words:
                if first and w == first.group(1) and sentence.strip().startswith(w):
                    continue                                    # sentence-initial: skip
                if w.rstrip(".") not in ALLOW:
                    candidates.append(w)
    return candidates

english_pseudo = corpus.loc[corpus["lang"] == "en", "pseudonymized"]
found = Counter(w for t in english_pseudo for w in leak_candidates(t))
print(found.most_common(15))
# Most of what this finds are real leaks: streets, villages and names that the NER missed, and month
# names that belong to unmasked dates of birth (we didn't add the DATE pattern from Exercise 1 here).
# A short list for a human to check is far quicker than rereading every transcript. Lowercase names
# (from speech recognition) slip through this check, so read a random sample in full as well.

## 8. Tools and good practice

### Tools worth knowing

- **Microsoft Presidio**: an open-source framework that combines patterns, NER models and context
  words, with ready-made recognizers for many identifier types and support for adding your own.
- **deduce**: de-identification of Dutch (medical) text, developed at UMC Utrecht (Menger et al.,
  2018).
- **Textwash**: an open-source anonymization tool evaluated with real re-identification attempts
  (Kleinberg et al., 2022).
- **Larger NER models**: `en_core_web_trf`, `nl_core_news_lg`, or a BERT model fine-tuned for NER
  (Day 3), catch noticeably more names than the small models used here.
- **Large language models** can find indirect identifiers that patterns miss, but sending
  confidential transcripts to an online API is exactly the data transfer you wanted to avoid. If you
  use one, use a model that runs locally, and still check the output.

### A workflow that holds up

1. Decide what must go (your DMP and consent forms), and whether pseudonymization is enough.
2. Run patterns + NER, with consistent pseudonyms; keep the key separate and secure.
3. Measure recall on a hand-checked sample of *your* data, per identifier type.
4. Generate a list of leak candidates and check it; read a random sample in full.
5. Handle indirect identifiers by hand: generalize, or leave details out.
6. Document what you did, so others can judge the remaining risk.

### Further reading

- Carrell, D., Malin, B., Aberdeen, J., Bayer, S., Clark, C., Wellner, B., & Hirschman, L. (2013).
  Hiding in plain sight: Use of realistic surrogates to reduce exposure of protected health
  information in clinical text. *JAMIA, 20*(2), 342-348.
- Kleinberg, B., Davies, T., & Mozes, M. (2022). Textwash: Automated open-source text anonymisation.
  *arXiv:2208.13081*.
- Menger, V., Scheepers, F., van Wijk, L. M., & Spruit, M. (2018). DEDUCE: A pattern matching method
  for automatic de-identification of Dutch medical text. *Telematics and Informatics, 35*(4), 727-736.